# INSIVUMEH · GOES-16 / 18 / 19 · Radiancia física C07

**Reproducción estricta del algoritmo solicitado: radiancia Level 1b, máximo − media espacial y media móvil temporal retrospectiva de 12 horas.**

Esta versión **NO** transforma radiancia en temperatura de brillo, potencia radiativa ni longitud de onda; **NO** introduce anillos de fondo, umbrales térmicos ni interpolación de huecos temporales. Emplea el canal C07 y el valor `Rad` de los archivos públicos `ABI-L1b-RadF` de NOAA/AWS.

**Flujo:** localizar archivos por fecha y satélite → descargar NetCDF automáticamente → leer `Rad` sin doble decodificación → recortar en la proyección GOES (sweep=x) → reproyectar bilinealmente a EPSG:4326 en Float32 → seleccionar centros de píxel mediante Haversine ≤ 12 km → calcular Lmax, Lmean, Lmin y D=Lmax−Lmean → filtrar período → media móvil 12 h retrospectiva → gráficas y CSV.

**Satélite:** automático GOES-East = GOES-16 antes de 2025-04-07 15:10 UTC y GOES-19 desde entonces; GOES-18 (West) disponible manualmente desde su entrada operacional de enero de 2023. NOAA: [GOES-East 19](https://www.noaa.gov/news-release/noaas-goes-19-satellite-now-operational-providing-critical-new-data-to-forecasters), [archivo abierto AWS](https://registry.opendata.aws/noaa-goes/).

**Importante:** las estadísticas dependen de extensión, resolución y reproyección. La media móvil es **temporal** y solo se calcula con observaciones finitas del período visible. La validación científica exige comprobar la concordancia espacial con el visor de referencia.

## 1 · Instalar dependencias en Google Colab

La descarga de los NetCDF ocurre **de NOAA hacia el entorno remoto de Colab**. Solo los archivos exportados viajan a tu dispositivo cuando eliges descargarlos.

In [19]:
%pip -q install netCDF4 rasterio pyproj numpy pandas matplotlib boto3 ipywidgets

## 2 · Bibliotecas, constantes y parámetros científicos

La fórmula del **radio Haversine** utiliza exactamente `6371.0088 km`. La salida usa `EPSG:4326`, `Float32`, `NoData = -9999` y grados configurables. Las series temporales nunca sustituyen ausencias por cero.

In [20]:
from pathlib import Path
from datetime import datetime, timedelta, timezone
from io import StringIO
import json, math, re, warnings, zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import rasterio
from rasterio.io import MemoryFile
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.transform import from_origin
from rasterio.windows import Window, from_bounds
from pyproj import CRS, Transformer
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import ipywidgets as W
from IPython.display import display, clear_output, FileLink

UTC=timezone.utc
RADIO_TIERRA_KM=6371.0088
NODATA=-9999.0
UNIDADES_GRAFICO='mW m⁻² sr⁻¹ (cm⁻¹)⁻¹'
COLOR_ORIGINAL='#1ab7bb'
COLOR_MEDIA_MOVIL='#c6b18c'
CORTE_GOES_EAST=datetime(2025,4,7,15,10,tzinfo=UTC)
GOES18_WEST_DESDE=datetime(2023,1,4,18,0,tzinfo=UTC)
DIR=Path('/content/INSIVUMEH_GOES_RADIANCIA' if Path('/content').exists() else '/mnt/data/INSIVUMEH_GOES_RADIANCIA')
DIR.mkdir(parents=True,exist_ok=True)
RAW=DIR/'NetCDF';RAW.mkdir(exist_ok=True)
RESULTADOS=DIR/'resultados';RESULTADOS.mkdir(exist_ok=True)
S3=boto3.client('s3',region_name='us-east-1',config=Config(signature_version=UNSIGNED,connect_timeout=15,read_timeout=90,retries={'max_attempts':4}))
print('Radiancia física GOES C07 | Sin conversión BT | Haversine R=6371.0088 km')

Radiancia física GOES C07 | Sin conversión BT | Haversine R=6371.0088 km


## 3 · Selección de satélite, fechas, grilla y volcanes

Valores iniciales para estudiar el episodio **2–3 de junio de 2018** con GOES-16. La extensión de ejemplo incluye Fuego, Pacaya y Santiaguito. **Max. escenas** protege recursos, pero **nunca** recorta silenciosamente el intervalo: si se supera el máximo, muestra una advertencia y pide ajustar ese límite o el paso temporal.

- `Paso mínimo`: separación mínima entre **escenas reales disponibles**, no genera horarios ficticios.
- Si seleccionas 60 minutos, las líneas se interrumpen entre puntos porque la regla exigida es cortar trazos con huecos **mayores de 30 minutos**.
- La resolución predeterminada es `0.02°`. No representa necesariamente la resolución espacial original del ABI.

In [21]:
sat=W.Dropdown(options=[('Automático GOES-East 16→19','auto'),('GOES-16 manual','16'),('GOES-18 manual (West)','18'),('GOES-19 manual','19')],value='auto',description='Satélite:',layout=W.Layout(width='430px'))
inicio=W.Text(value='2018-06-02T12:00:00',description='Inicio UTC:',layout=W.Layout(width='320px'))
fin=W.Text(value='2018-06-03T23:00:00',description='Fin UTC:',layout=W.Layout(width='320px'))
paso=W.Dropdown(options=[('10 min',10),('15 min',15),('20 min',20),('30 min',30),('60 min',60)],value=15,description='Paso min:')
max_escenas=W.IntText(value=400,description='Máx. escenas:',layout=W.Layout(width='230px'))
oeste=W.FloatText(value=-92.10,description='Oeste:')
sur=W.FloatText(value=13.65,description='Sur:')
este=W.FloatText(value=-90.05,description='Este:')
norte=W.FloatText(value=15.55,description='Norte:')
resol=W.FloatText(value=.02,description='Resol. °:')
volcanes_widgets={}
for nom,latitude,longitude in [('Fuego',14.47459,-90.88074),('Pacaya',14.38249,-90.60136),('Santiaguito',14.73938,-91.56816)]:
    la=W.FloatText(value=latitude,description='lat:',layout=W.Layout(width='190px'))
    lo=W.FloatText(value=longitude,description='lon:',layout=W.Layout(width='190px'))
    ra=W.FloatText(value=12.0,description='radius_km:',layout=W.Layout(width='240px'))
    volcanes_widgets[nom]=(la,lo,ra)
conservar_nc=W.Checkbox(value=False,description='Conservar NetCDF descargados')
guardar_tif=W.Checkbox(value=False,description='Guardar GeoTIFF de cada escena (más espacio)')
ejecutar=W.Button(description='PROCESAR RADIANCIA FÍSICA Y MEDIA MÓVIL 12 h',button_style='success',layout=W.Layout(width='430px'))
salida=W.Output()
secciones=[W.HTML('<h3>NOAA GOES C07 · Rad física Level 1b (sin BT)</h3>'),sat,W.HBox([inicio,fin]),W.HBox([paso,max_escenas]),W.HTML('<b>Extensión de salida EPSG:4326 (grados)</b>'),W.HBox([oeste,sur,este]),W.HBox([norte,resol]),W.HTML('<b>Centros de volcanes y radios (km) — 12 km iniciales</b>')]
for nom,(la,lo,ra) in volcanes_widgets.items():
    secciones.extend([W.HTML('<b>'+nom+'</b>'),W.HBox([la,lo,ra])])
secciones.extend([W.HBox([conservar_nc,guardar_tif]),ejecutar,salida])
display(W.VBox(secciones))

def utc_parse(text):
    t=pd.Timestamp(str(text).strip())
    if t.tzinfo is None:t=t.tz_localize('UTC')
    return t.tz_convert('UTC').to_pydatetime()

def configuracion():
    cfg={'satelite':sat.value,'inicio':utc_parse(inicio.value),'fin':utc_parse(fin.value),
         'paso_min':int(paso.value),'max_escenas':int(max_escenas.value),
         'bbox':(float(oeste.value),float(sur.value),float(este.value),float(norte.value)),
         'resol_deg':float(resol.value),
         'volcanes':{nombre:{'lat':float(w[0].value),'lon':float(w[1].value),'radius_km':float(w[2].value)} for nombre,w in volcanes_widgets.items()},
         'conservar_nc':bool(conservar_nc.value),'guardar_tif':bool(guardar_tif.value)}
    w,s,e,n=cfg['bbox']
    if not cfg['inicio']<cfg['fin']:raise ValueError('Inicio UTC debe ser anterior a Fin UTC')
    if not (-180<=w<e<=180 and -90<=s<n<=90):raise ValueError('La extensión geográfica es inválida')
    if not (.001<=cfg['resol_deg']<=.2):raise ValueError('La resolución debe estar entre 0.001° y 0.2°')
    if cfg['max_escenas']<1:raise ValueError('Máx. escenas debe ser mayor que cero')
    if (e-w)*(n-s)/cfg['resol_deg']**2>3_000_000:raise ValueError('Grilla demasiado grande; reduzca región o aumente la resolución en grados')
    for nombre,v in cfg['volcanes'].items():
        if not(w<=v['lon']<=e and s<=v['lat']<=n):raise ValueError(f'{nombre} queda fuera de la región de salida')
        if v['radius_km']<=0:raise ValueError(f'El radio de {nombre} debe ser positivo')
    return cfg

## 4 · Buscar exclusivamente archivos originales Level 1b C07

Se consulta S3 público `noaa-goes16`, `noaa-goes18` o `noaa-goes19`. El nombre correcto contiene **`ABI-L1b-RadF` y `C07`**. Se filtran las fechas de adquisición reales para **no crear escenas inexistentes**, algo relevante con GOES-16 en junio de 2018 (cadencia Full Disk aproximada de 15 min).

En automático, el cambio GOES-16→GOES-19 ocurre por fecha **de cada archivo**, incluso si el intervalo cruza 2025-04-07. GOES-18 no se mezcla automáticamente con GOES-East.

In [22]:
def satelite_para(t, modo='auto'):
    if modo=='auto':return '16' if t<CORTE_GOES_EAST else '19'
    return str(modo)

def leer_fecha_nombre(filename):
    m=re.search(r'_s(\d{4})(\d{3})(\d{2})(\d{2})(\d{2})(\d{1,6})_',filename)
    if not m:raise ValueError('Nombre GOES sin tiempo de inicio: '+filename)
    year,doy,hour,minute,second,decimal=m.groups()
    return datetime(int(year),1,1,tzinfo=UTC)+timedelta(days=int(doy)-1,hours=int(hour),minutes=int(minute),seconds=int(second),microseconds=int(decimal.ljust(6,'0')))

def _horas_utc(start,end):
    t=start.replace(minute=0,second=0,microsecond=0)
    while t<=end:
        yield t
        t+=timedelta(hours=1)

def buscar_escenas(cfg):
    encontrados={}
    for hora in _horas_utc(cfg['inicio'],cfg['fin']):
        # En la hora del relevo de abril 2025 se consultan los dos satélites.
        if cfg['satelite']=='auto':
            ids={satelite_para(hora),satelite_para(hora+timedelta(hours=1,microseconds=-1))}
        else:ids={cfg['satelite']}
        for numero in sorted(ids):
            bucket='noaa-goes'+numero
            prefix=f'ABI-L1b-RadF/{hora.year}/{hora.timetuple().tm_yday:03d}/{hora.hour:02d}/'
            pag=S3.get_paginator('list_objects_v2')
            try:
                for page in pag.paginate(Bucket=bucket,Prefix=prefix):
                    for obj in page.get('Contents',[]):
                        key=obj['Key'];fname=Path(key).name
                        if not re.search(r'ABI-L1b-RadF-M\d+C07_G'+numero+r'_',fname):continue
                        try:tm=leer_fecha_nombre(fname)
                        except ValueError:continue
                        if not cfg['inicio']<=tm<=cfg['fin']:continue
                        if cfg['satelite']=='auto' and satelite_para(tm)!=numero:continue
                        encontrados[(numero,tm)]={'numero':numero,'bucket':bucket,'key':key,'fecha_objeto':tm,'size':int(obj.get('Size',0))}
            except Exception as exc:
                raise RuntimeError(f'Error de acceso a NOAA/AWS: {bucket}/{prefix}: {exc}') from exc
    rows=sorted(encontrados.values(),key=lambda x:x['fecha_objeto'])
    # Selecciona escenas existentes que mantienen al menos N minutos entre sí.
    seleccion=[]
    for row in rows:
        if not seleccion or (row['fecha_objeto']-seleccion[-1]['fecha_objeto']).total_seconds()>=cfg['paso_min']*60-1:
            seleccion.append(row)
    if len(seleccion)>cfg['max_escenas']:
        raise ValueError(f'Se encontraron {len(seleccion)} escenas reales; el máximo actual es {cfg["max_escenas"]}. No se recortará el intervalo: aumenta Máx. escenas o el paso mínimo (min).')
    return seleccion

def descargar_nc(row):
    destino=RAW/Path(row['key']).name
    if destino.exists() and (row['size']==0 or destino.stat().st_size==row['size']):return destino
    temporal=destino.with_suffix('.nc.part')
    temporal.unlink(missing_ok=True)
    try:
        S3.download_file(row['bucket'],row['key'],str(temporal))
        if row['size'] and temporal.stat().st_size!=row['size']:raise IOError('Descarga incompleta: tamaño diferente al inventario S3')
        temporal.replace(destino)
    finally:temporal.unlink(missing_ok=True)
    return destino

print('Inventario: solo ABI-L1b-RadF-C07; las fechas de adquisición proceden de nombres reales NOAA.')

Inventario: solo ABI-L1b-RadF-C07; las fechas de adquisición proceden de nombres reales NOAA.


## 5 · Leer `Rad` y aplicar escala **una sola vez**

Utilizamos `netCDF4.Dataset` y desactivamos expresamente la decodificación automática **solo para `Rad`** (`set_auto_maskandscale(False)`). Así se tiene acceso al valor empaquetado y a `_FillValue` originales. Después se calcula **una sola vez**:

`L = valor_empaquetado × scale_factor + add_offset`

Se excluyen valores de relleno **antes** de escalar. Se conservan los metadatos originales de radiancia, proyección y tiempo en `metadatos_escenas.json`. No se aplica Planck ni otra transformación de unidades.

In [23]:
def _python_val(v):
    if isinstance(v,np.ndarray):return v.tolist()
    if isinstance(v,np.generic):return v.item()
    if isinstance(v,bytes):return v.decode('utf8','replace')
    return v

def obtener_meta_y_radiancia(path, bbox):
    from netCDF4 import Dataset
    with Dataset(str(path),'r') as nc:
        if 'Rad' not in nc.variables:raise KeyError('Falta variable Rad: archivo no compatible')
        if 'goes_imager_projection' not in nc.variables:raise KeyError('Falta goes_imager_projection')
        p=nc.variables['goes_imager_projection']
        pmeta={a:_python_val(p.getncattr(a)) for a in p.ncattrs()}
        h=float(pmeta['perspective_point_height'])
        lon0=float(pmeta['longitude_of_projection_origin'])
        a=float(pmeta['semi_major_axis']);b=float(pmeta['semi_minor_axis'])
        sweep=pmeta.get('sweep_angle_axis','x')
        if sweep!='x':raise ValueError('La proyección no tiene sweep=x')
        # h es la altura sobre la superficie del elipsoide: NO se le suma el radio terrestre.
        geos=CRS.from_proj4(f'+proj=geos +lon_0={lon0} +h={h} +a={a} +b={b} +sweep=x +units=m +no_defs')
        to_goes=Transformer.from_crs('EPSG:4326',geos,always_xy=True)
        w,s,e,n=bbox
        gx=np.r_[np.linspace(w,e,41),np.full(41,w),np.full(41,e),np.linspace(w,e,41)]
        gy=np.r_[np.full(41,s),np.linspace(s,n,41),np.linspace(s,n,41),np.full(41,n)]
        X,Y=to_goes.transform(gx,gy)
        valid=np.isfinite(X)&np.isfinite(Y)
        if not valid.any():raise ValueError('La región está fuera de la imagen de este satélite')
        xvar=nc.variables['x'];yvar=nc.variables['y']
        # x/y se leen con la decodificación CF habitual (son ángulos de barrido en radianes).
        x=np.asarray(np.ma.array(xvar[:],dtype='float64').filled(np.nan))*h
        y=np.asarray(np.ma.array(yvar[:],dtype='float64').filled(np.nan))*h
        difx=float(np.nanmedian(np.abs(np.diff(x))));dify=float(np.nanmedian(np.abs(np.diff(y))))
        mx=(x>=np.nanmin(X[valid])-3*difx)&(x<=np.nanmax(X[valid])+3*difx)
        my=(y>=np.nanmin(Y[valid])-3*dify)&(y<=np.nanmax(Y[valid])+3*dify)
        if not mx.any() or not my.any():raise ValueError('La región no se cruza con la grilla GOES')
        x0,x1=int(np.where(mx)[0][0]),int(np.where(mx)[0][-1]+1)
        y0,y1=int(np.where(my)[0][0]),int(np.where(my)[0][-1]+1)
        rad=nc.variables['Rad']
        dtype_original=str(rad.dtype)
        attrs={key:_python_val(rad.getncattr(key)) for key in rad.ncattrs() if key in ['units','scale_factor','add_offset','_FillValue','missing_value','valid_range','long_name']}
        # Desactivar la decodificación automática para impedir doble escala/offset.
        rad.set_auto_maskandscale(False)
        raw=np.asarray(rad[y0:y1,x0:x1])
        valid_raw=np.isfinite(raw)
        for atributo in ['_FillValue','missing_value']:
            if atributo in attrs:
                for f in np.atleast_1d(attrs[atributo]):valid_raw&=(raw!=f)
        if 'valid_range' in attrs:
            rng=np.atleast_1d(attrs['valid_range'])
            if len(rng)==2:valid_raw&=(raw>=rng[0])&(raw<=rng[1])
        factor=float(attrs.get('scale_factor',1.0))
        offset=float(attrs.get('add_offset',0.0))
        # Exactamente una conversión de los valores empaquetados a radiancia física.
        L=np.asarray(raw,dtype='float64')*factor+offset
        L[~valid_raw]=np.nan
        data=np.where(np.isfinite(L),L,NODATA).astype('float32')
        xx=x[x0:x1];yy=y[y0:y1]
        if xx.size<2 or yy.size<2:raise ValueError('No hay suficientes píxeles fuente')
        if xx[0]>xx[-1]:xx=xx[::-1];data=data[:,::-1]
        if yy[0]<yy[-1]:yy=yy[::-1];data=data[::-1,:]
        dx=float(np.nanmedian(np.diff(xx)));dy=float(np.nanmedian(np.diff(yy)))
        if dx<=0 or dy>=0:raise ValueError('Ejes de la proyección geostacionaria inválidos')
        trans=from_origin(float(xx[0]-dx/2),float(yy[0]+abs(dy)/2),float(dx),float(abs(dy)))
        t0=nc.getncattr('time_coverage_start') if 'time_coverage_start' in nc.ncattrs() else None
        t1=nc.getncattr('time_coverage_end') if 'time_coverage_end' in nc.ncattrs() else None
        tvar=nc.variables.get('t')
        tmeta=({'unidades':_python_val(getattr(tvar,'units','')),'valor':_python_val(np.asarray(tvar[:]).tolist())} if tvar is not None else {})
        metadata={'archivo':Path(path).name,'Rad_attrs':attrs,'Rad_dtype_original':dtype_original,
                  'Rad_aplicacion_escala':'manual exactamente una vez; auto_maskandscale(False)',
                  'proyeccion':pmeta,'fecha_UTC_metadato':t0,'fin_UTC_metadato':t1,'t':tmeta,
                  'fuente_nodata':int((~valid_raw).sum()),'fuente_ventana':[y0,y1,x0,x1]}
        return data,trans,geos,metadata

print('Rad: lectura empaquetada; scale_factor/add_offset aplicados una sola vez.')

Rad: lectura empaquetada; scale_factor/add_offset aplicados una sola vez.


## 6 · Reproyectar a EPSG:4326 con GDAL y seleccionar círculos de Haversine

Primero se recorta una ventana en la **proyección nativa GOES**. Su transformada usa `x × perspective_point_height` y `y × perspective_point_height`, como pide la proyección fija del ABI. Después GDAL/Rasterio aplica **interpolación bilineal** a la grilla de salida (`0.02°` por defecto) en `Float32`, `NoData=-9999`.

Sobre **esa grilla reproyectada** se extrae una ventana rectangular por volcán y se toman únicamente centros de píxel a ≤ `radius_km`, calculados mediante la distancia de Haversine y radio terrestre **6371.0088 km**. No hay ponderación por superficie ni por fracción de píxel. Se excluyen NoData, valores infinitos y píxeles enmascarados por GDAL.

In [24]:
def haversine_km(lat0,lon0,lat1,lon1):
    p0=np.deg2rad(lat0);p1=np.deg2rad(lat1)
    dlat=np.deg2rad(lat1-lat0);dlon=np.deg2rad(lon1-lon0)
    a=np.sin(dlat/2)**2+np.cos(p0)*np.cos(p1)*np.sin(dlon/2)**2
    return 2*RADIO_TIERRA_KM*np.arcsin(np.sqrt(np.clip(a,0,1)))

def info_grilla(cfg):
    w,s,e,n=cfg['bbox'];res=cfg['resol_deg']
    width=int(math.ceil((e-w)/res));height=int(math.ceil((n-s)/res))
    return from_origin(w,n,res,res),width,height

def ventana_volcan(transform,width,height,v):
    lat,lon,r=v['lat'],v['lon'],v['radius_km']
    dlat=r/111.195;dlon=r/(111.195*max(0.05,np.cos(np.deg2rad(lat))))
    # Un margen permite tomar todos los centros que podrían quedar dentro de la circunferencia.
    raw=from_bounds(lon-dlon,lat-dlat,lon+dlon,lat+dlat,transform)
    x0=max(0,int(math.floor(raw.col_off)));y0=max(0,int(math.floor(raw.row_off)))
    x1=min(width,int(math.ceil(raw.col_off+raw.width)))
    y1=min(height,int(math.ceil(raw.row_off+raw.height)))
    if x1<=x0 or y1<=y0:return None
    return Window(x0,y0,x1-x0,y1-y0)

def estadisticas_desde_vrt(vrt,transform,width,height,volcan):
    window=ventana_volcan(transform,width,height,volcan)
    if window is None:return {'N':0,'Lmax':np.nan,'Lmean':np.nan,'Lmin':np.nan,'D':np.nan}
    arr=vrt.read(1,window=window,masked=True)
    vals=np.asarray(np.ma.getdata(arr),dtype='float32')
    gdalmask=vrt.read_masks(1,window=window)>0
    # La máscara de dataset GDAL puede excluir píxeles adicionales.
    datasetmask=vrt.dataset_mask(window=window)>0
    rows=window.row_off+np.arange(int(window.height))+0.5
    cols=window.col_off+np.arange(int(window.width))+0.5
    lonvec=transform.c+cols*transform.a
    latvec=transform.f+rows*transform.e
    xx,yy=np.meshgrid(lonvec,latvec)
    dist=haversine_km(volcan['lat'],volcan['lon'],yy,xx)
    valid=(dist<=volcan['radius_km'])&gdalmask&datasetmask&(~np.ma.getmaskarray(arr))&np.isfinite(vals)&(vals!=NODATA)
    seleccion=vals[valid]
    n=int(seleccion.size)
    if not n:return {'N':0,'Lmax':np.nan,'Lmean':np.nan,'Lmin':np.nan,'D':np.nan}
    maximum=float(np.max(seleccion))
    mean=float(np.sum(seleccion,dtype='float64')/n)
    minimum=float(np.min(seleccion))
    return {'N':n,'Lmax':maximum,'Lmean':mean,'Lmin':minimum,'D':maximum-mean}

def estadisticas_escena(path,cfg,ruta_tif=None):
    srcdata,srctrans,srccrs,metadata=obtener_meta_y_radiancia(path,cfg['bbox'])
    transform,width,height=info_grilla(cfg)
    with MemoryFile() as mem:
        with mem.open(driver='GTiff',height=srcdata.shape[0],width=srcdata.shape[1],count=1,
                      dtype='float32',crs=srccrs,transform=srctrans,nodata=NODATA) as writer:
            writer.write(srcdata,1)
            writer.write_mask(np.where(np.isfinite(srcdata)&(srcdata!=NODATA),255,0).astype('uint8'))
        with mem.open() as src:
            with WarpedVRT(src,crs='EPSG:4326',transform=transform,width=width,height=height,
                           resampling=Resampling.bilinear,src_nodata=NODATA,nodata=NODATA,
                           dtype='float32',init_dest_nodata=True) as vrt:
                res={nombre:estadisticas_desde_vrt(vrt,transform,width,height,v) for nombre,v in cfg['volcanes'].items()}
                if ruta_tif is not None:
                    # Guardar la grilla física ya reproyectada, sin conversiones de magnitud.
                    arr=vrt.read(1,masked=True)
                    values=np.where(np.ma.getmaskarray(arr),NODATA,np.asarray(np.ma.getdata(arr),dtype='float32'))
                    with rasterio.open(ruta_tif,'w',driver='GTiff',height=height,width=width,count=1,
                                       dtype='float32',crs='EPSG:4326',transform=transform,nodata=NODATA,
                                       compress='deflate') as tif:
                        tif.write(values.astype('float32'),1)
                        tif.write_mask(vrt.dataset_mask())
    return res,metadata

print('Círculo: centros de píxel en EPSG:4326, distancia Haversine R=6371.0088 km.')

Círculo: centros de píxel en EPSG:4326, distancia Haversine R=6371.0088 km.


## 7 · Fórmulas de la curva y de la media móvil temporal

Por cada escena y volcán:

- `Lmax(t) = max(Li)`
- `Lmean(t) = sum(Li) / N`
- `Lmin(t) = min(Li)` — guardada en CSV; **no** se grafica.
- **`D(t) = Lmax(t) − Lmean(t)`**, etiqueta **«Máxima - Media»**.

Para la media móvil retrospectiva:

`M12h(t) = promedio de los D(tj) finitos para t−12 h ≤ tj ≤ t`.

Primero **filtramos el período elegido** y luego calculamos `rolling('12h', closed='both', min_periods=1)`. No interpolamos huecos, no ponderamos por tiempo y no exigimos 12 horas completas. **Advertencia**: al principio del período la media utiliza solo el contexto disponible dentro de la selección.

In [25]:
def construir_series(registros,inicio_utc,fin_utc):
    df=pd.DataFrame(registros)
    if df.empty:return df
    df['fecha_UTC']=pd.to_datetime(df['fecha_UTC'],format='ISO8601',utc=True,errors='coerce')
    df=df.dropna(subset=['fecha_UTC'])
    df=df[(df['fecha_UTC']>=pd.Timestamp(inicio_utc))&(df['fecha_UTC']<=pd.Timestamp(fin_utc))]
    df=df.sort_values(['volcan','fecha_UTC']).reset_index(drop=True)
    df['M12h']=np.nan
    for nombre,inds in df.groupby('volcan',sort=False).groups.items():
        tmp=df.loc[inds].sort_values('fecha_UTC')
        d=pd.to_numeric(tmp['D'],errors='coerce')
        # La media móvil se calcula sobre las fechas reales, no un número fijo de filas.
        d.index=pd.DatetimeIndex(tmp['fecha_UTC'])
        mov=d.rolling('12h',closed='both',min_periods=1).mean()
        df.loc[tmp.index,'M12h']=mov.to_numpy()
    return df

print('M12h: 12 h retrospectivas inclusivas. N disponible, sin relleno de huecos.')

M12h: 12 h retrospectivas inclusivas. N disponible, sin relleno de huecos.


## 8 · Gráficas por volcán: unidades y huecos temporales

La gráfica presenta **solo dos curvas**: **Máxima - Media** en turquesa `#1ab7bb` y **Media móvil 12h** en arena `#c6b18c`. El eje Y usa exclusivamente `mW m⁻² sr⁻¹ (cm⁻¹)⁻¹`. La serie **se interrumpe si pasan más de 30 minutos** entre dos mediciones consecutivas válidas; se conservan puntos aislados sin unirlos. No se grafica `Lmin`.

In [26]:
def dibujar_segmentos(ax, fechas, valores, color, etiqueta, gap_min=30):
    fechas=pd.to_datetime(fechas,format='ISO8601',utc=True)
    y=np.asarray(valores,dtype='float64')
    # No dibujar M12h en timestamps donde la observación de D está ausente (para no simular medición).
    valid=np.isfinite(y)
    indices=np.where(valid)[0]
    if not len(indices):
        ax.plot([],[],color=color,lw=1.8,label=etiqueta)
        return
    cortes=[0]
    for j in range(1,len(indices)):
        anterior=indices[j-1];actual=indices[j]
        gap=(fechas.iloc[actual]-fechas.iloc[anterior]).total_seconds()/60
        if actual!=anterior+1 or gap>gap_min:cortes.append(j)
    cortes.append(len(indices))
    primera=True
    for a,b in zip(cortes[:-1],cortes[1:]):
        ix=indices[a:b]
        if len(ix)>1:
            ax.plot(fechas.iloc[ix].to_numpy(),y[ix],color=color,lw=1.8,label=etiqueta if primera else '_nolegend_')
            primera=False
    ax.scatter(fechas.iloc[indices],y[indices],color=color,s=9,zorder=4,label=etiqueta if primera else '_nolegend_')

def graficar_volcan(df,nombre,folder,cfg):
    subset=df[df['volcan']==nombre].sort_values('fecha_UTC').copy()
    if subset.empty:return None
    fig,ax=plt.subplots(figsize=(13,5.4),dpi=120)
    dibujar_segmentos(ax,subset['fecha_UTC'],subset['D'],COLOR_ORIGINAL,'Máxima - Media')
    # La media móvil respeta la ausencia de una observación presente en el cálculo de D.
    mm=subset['M12h'].where(subset['D'].notna(),np.nan)
    dibujar_segmentos(ax,subset['fecha_UTC'],mm,COLOR_MEDIA_MOVIL,'Media móvil 12h')
    ax.set_title(f'{nombre} · GOES C07 Level 1b · Radiancia física',fontsize=14,weight='bold')
    ax.set_xlabel('Fecha y hora UTC')
    ax.set_ylabel(UNIDADES_GRAFICO)
    ax.grid(alpha=.22);ax.legend(loc='best')
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=4,maxticks=9))
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator(),tz=UTC))
    fig.text(.5,.01,'D = Lmax − Lmean espacial | Media móvil retrospectiva 12 h | Huecos > 30 min interrumpen trazos',ha='center',fontsize=9,color='#595959')
    fig.tight_layout(rect=[0,.035,1,1])
    name=re.sub(r'[^a-zA-Z0-9]+','_',nombre).strip('_')
    png=folder/f'{name}_C07_Maxima_menos_Media_y_Media_Movil_12h.png'
    svg=png.with_suffix('.svg')
    fig.savefig(png,dpi=150,bbox_inches='tight');fig.savefig(svg,bbox_inches='tight')
    plt.close(fig)
    return png,svg

print('Trazos: no conectan intervalos superiores a 30 min; se exporta PNG y SVG.')

Trazos: no conectan intervalos superiores a 30 min; se exporta PNG y SVG.


## 9 · Ejecución completa, CSV y ZIP

Presiona el botón verde del formulario de la sección 3. El programa consulta todos los horas relevantes, **no limita la salida a las primeras 24 escenas**, registra errores por escena y genera una fila CSV por volcán y observación. Los campos nulos indican ausencia de datos (no cero).

CSV principal: `fecha_UTC`, `satelite`, `volcan`, `lat`, `lon`, `radius_km`, `N`, `Lmax`, `Lmean`, `Lmin`, `D`, `M12h`, `unidades`, `archivo`, `estado`.

Metadatos adicionales: `Rad_attrs`, proyección nativa, altura, sweep, escala, offset, valor de relleno, cobertura temporal y operación de decodificación.

In [27]:
def _json_default(obj):
    if isinstance(obj,(np.floating,np.integer)):return obj.item()
    if isinstance(obj,(datetime,pd.Timestamp)):return obj.isoformat()
    if isinstance(obj,Path):return str(obj)
    raise TypeError(type(obj).__name__)

def procesar(cfg):
    stamp=cfg['inicio'].strftime('%Y%m%d_%H%M')+'_'+cfg['fin'].strftime('%Y%m%d_%H%M')
    folder=RESULTADOS/('C07_RADIANCIA_'+stamp);folder.mkdir(parents=True,exist_ok=True)
    print('Buscando escenas en NOAA (GOES '+cfg['satelite']+')...')
    escenas=buscar_escenas(cfg)
    if not escenas:raise FileNotFoundError('No hay archivos ABI-L1b-RadF C07 en el período/satélite solicitados')
    total_gb=sum(x['size'] for x in escenas)/1e9
    print(f'Escenas reales seleccionadas: {len(escenas)} | Descarga estimada: {total_gb:.2f} GB')
    print('No hay recorte temporal silencioso. Descarga en Colab, no directamente a tu computadora.')
    regs=[];log=[];metadatos=[]
    tifdir=folder/'GeoTIFF_EPSG4326'
    if cfg['guardar_tif']:tifdir.mkdir(exist_ok=True)
    for i,row in enumerate(escenas,1):
        tm=row['fecha_objeto'];local=None
        print(f'[{i}/{len(escenas)}] GOES-{row["numero"]} {tm:%Y-%m-%d %H:%M:%S} UTC',end=' ')
        try:
            local=descargar_nc(row)
            ruta_tif=tifdir/(Path(row['key']).stem+'.tif') if cfg['guardar_tif'] else None
            stats,meta=estadisticas_escena(local,cfg,ruta_tif)
            timestamp=meta.get('fecha_UTC_metadato')
            # Si el metadato es ambiguo, recurrir a la hora real codificada en el nombre.
            try:tiempo=utc_parse(timestamp) if timestamp else tm
            except Exception:tiempo=tm
            metadatos.append(meta)
            for nombre,v in cfg['volcanes'].items():
                es=stats[nombre]
                regs.append({'fecha_UTC':tiempo.isoformat(),'satelite':'GOES-'+row['numero'],'volcan':nombre,
                    'lat':v['lat'],'lon':v['lon'],'radius_km':v['radius_km'],
                    **es,'M12h':np.nan,'unidades':meta['Rad_attrs'].get('units','no declarado'),
                    'archivo':Path(row['key']).name,'estado':'OK' if es['N'] else 'sin_pixeles_validos'})
            log.append({'fecha_UTC':tiempo.isoformat(),'satelite':'GOES-'+row['numero'],'archivo':Path(row['key']).name,'estado':'OK','detalle':''})
            print('OK | N: '+', '.join(f'{k}={v["N"]}' for k,v in stats.items()))
        except Exception as exc:
            log.append({'fecha_UTC':tm.isoformat(),'satelite':'GOES-'+row['numero'],'archivo':Path(row['key']).name,'estado':'ERROR','detalle':str(exc)})
            for nombre,v in cfg['volcanes'].items():
                regs.append({'fecha_UTC':tm.isoformat(),'satelite':'GOES-'+row['numero'],'volcan':nombre,
                    'lat':v['lat'],'lon':v['lon'],'radius_km':v['radius_km'],
                    'N':0,'Lmax':np.nan,'Lmean':np.nan,'Lmin':np.nan,'D':np.nan,'M12h':np.nan,
                    'unidades':UNIDADES_GRAFICO,'archivo':Path(row['key']).name,'estado':'error: '+str(exc)[:180]})
            print('ERROR:',str(exc)[:220])
        finally:
            if local is not None and not cfg['conservar_nc']:
                try:local.unlink(missing_ok=True)
                except OSError:pass
    df=construir_series(regs,cfg['inicio'],cfg['fin'])
    path_csv=folder/'RADIANCIA_GOES_C07_TODOS_LOS_VOLCANES.csv'
    df.to_csv(path_csv,index=False,na_rep='',float_format='%.10g')
    pd.DataFrame(log).to_csv(folder/'registro_escenas.csv',index=False)
    for nombre in cfg['volcanes']:
        df[df['volcan']==nombre].to_csv(folder/(nombre+'_C07_estadisticas.csv'),index=False,na_rep='',float_format='%.10g')
        result=graficar_volcan(df,nombre,folder,cfg)
        if result:print('Gráfica:',nombre,'→',result[0].name)
    settings={**cfg,'fuente':'NOAA ABI-L1b-RadF-C07','banda':'Rad','proyeccion_salida':'EPSG:4326',
             'tipo':'Float32','NoData':NODATA,'interpolacion':'bilineal GDAL','radio_tierra_km':RADIO_TIERRA_KM,
             'formula_D':'Lmax - Lmean','formula_M12h':'media retrospectiva de D finito, [t-12h,t], extremos incluidos',
             'restriccion':'primero filtrar periodo elegido; no usa datos anteriores al inicio',
             'corte_dibujo_huecos_min':30,'grafica_Lmin':False}
    (folder/'configuracion_y_metodologia.json').write_text(json.dumps(settings,ensure_ascii=False,indent=2,default=_json_default),encoding='utf-8')
    (folder/'metadatos_escenas.json').write_text(json.dumps(metadatos,ensure_ascii=False,indent=2,default=_json_default),encoding='utf-8')
    (folder/'LEEME.txt').write_text('INSIVUMEH | GOES C07 radiancia física. L=raw*scale_factor+add_offset UNA VEZ. '+
      'Sobre grilla EPSG:4326 bilineal Float32 NoData -9999. D=Lmax-Lmean espacial. '+
      'M12h: media retrospectiva de observaciones finitas de D entre t-12h y t incluidos. '+
      'El filtrado temporal antecede a la media móvil, por lo que el inicio carece de contexto previo. '+
      'No se calcula BT ni potencia. Lmin conservada solo CSV. Las líneas no cruzan huecos mayores a 30 min.\n',encoding='utf-8')
    zipf=folder/'GOES_C07_RADIANCIA_FISICA_12H.zip'
    with zipfile.ZipFile(zipf,'w',zipfile.ZIP_DEFLATED) as z:
        for file in folder.iterdir():
            if file.is_file() and file!=zipf:z.write(file,file.name)
    exitos=sum(1 for e in log if e['estado']=='OK')
    print(f'Finalizado. Escenas correctas: {exitos}/{len(escenas)}. CSV: {path_csv.name}.')
    print('Lmin se incluye en CSV pero NO en las gráficas.')
    print('ZIP:',zipf)
    return {'tabla':df,'carpeta':folder,'zip':zipf,'registro':pd.DataFrame(log)}

ultima_ejecucion=None

def _al_pulsar(_):
    global ultima_ejecucion
    ejecutar.disabled=True
    with salida:
        clear_output(wait=True)
        try:
            cfg=configuracion()
            ultima_ejecucion=procesar(cfg)
            display(ultima_ejecucion['tabla'].head(10))
            try:
                from google.colab import files
                files.download(str(ultima_ejecucion['zip']))
            except ImportError:
                display(FileLink(str(ultima_ejecucion['zip'])))
        except Exception as exc:
            print('No se completó el procesamiento:',repr(exc))
            print('Revisa las fechas UTC, el número máximo de escenas y los mensajes específicos. No se añaden datos inventados.')
        finally:ejecutar.disabled=False

ejecutar.on_click(_al_pulsar)
print('Preparado: configura fecha/volcanes y pulsa el botón verde en la sección 3.')

Preparado: configura fecha/volcanes y pulsa el botón verde en la sección 3.


## 10 · Criterios de interpretación y verificaciones

- **No confundir instrumentos y magnitudes:** esta curva representa `Lmax−Lmean` de **radiancia Level 1b**. No es una curva de temperatura de brillo (BT), ni una estimación de lava, ni un índice MODVOLC.
- **No confundir medias:** `Lmean` es una media **espacial** de píxeles en una escena; `M12h` es una media **temporal** de diferencias de escenas previas.
- **Límites de la ventana:** la media móvil comienza con las observaciones del período mostrado. Si se desean valores basados en las 12 horas anteriores al inicio, habría que cambiar el criterio; deliberadamente **no** se hace aquí.
- **Huecos:** no se interpolan ni se rellenan. Se mantienen valores vacíos en el CSV y trazos separados si la distancia entre observaciones válidas supera 30 min.
- **Comparabilidad:** cambiar `0.02°`, el rectángulo, el centro, el radio o la órbita GOES puede alterar los píxeles seleccionados y las estadísticas; comparar solo configuraciones equivalentes.
- **Control de calidad:** para reproducir exactamente resultados históricos también deben concordar versiones de datos fuente, alineación y método de la grilla de salida, máscara GDAL e intervalos de tiempo.

**Comprobación recomendada en clase:** seleccionar el evento 2018-06-02 a 2018-06-03, GOES-16 automático, resolución 0.02°, radio 12 km; comprobar que en el CSV exista `Lmin`, que la curva original sea `Lmax−Lmean`, que la ventana temporal sea de 12 horas reales y que no se unan trazos a través de huecos de más de 30 minutos.